# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*


**Finding 1:** "Content refreshes leveraging structured data see a 40% higher organic visibility lift compared to unstructured rewrites."


* **Methodology Question:** How exactly is the "visibility lift" label isolated? Does the validation design control for concurrent site-wide authority changes, or could domains that implement structured data simply be higher-authority sites that grow faster naturally?


**Finding 2:** "AI-driven metadata optimization improves CTR by 15% on average within the first two weeks."


* **Methodology Question:** Is the baseline CTR established using a true A/B holdout group, or is this a simple pre/post time-series comparison? If it's pre/post, how does the methodology account for the natural seasonality of the search queries during those two weeks?

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

# 1. Loading Data
url = 'https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(url)

# 2. Setting up Target & Grouping (Same as Week 5)
pos_col, ctr_col, imp_col = 'avg_position', 'ctr', 'impressions_last_30d'
expected_ctr = np.where(df[pos_col] <= 3, 0.08, np.where(df[pos_col] <= 10, 0.03, 0.01))
y = ((df[imp_col] > df[imp_col].median()) & (df[ctr_col] < expected_ctr * 0.75) & (df[pos_col] <= 20)).astype(int)
groups = pd.qcut(df[imp_col].rank(method='first'), q=20, labels=False)

feature_cols = [c for c in df.select_dtypes(include=[np.number]).columns if 'id' not in c.lower() and c not in ['refresh_opportunity_label', 'synthetic_client_group', 'position_bucket', 'impression_bucket']]
X = df[feature_cols].fillna(0)

def precision_at_k(y_true, preds, k=50):
    top_k_idx = np.argsort(preds)[::-1][:k]
    return float(np.mean(np.asarray(y_true)[top_k_idx]))

clf = RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42)

# THE LEAKY SPLIT (Random Row Split)
X_train_rand, X_test_rand, y_train_rand, y_test_rand = train_test_split(X, y, test_size=0.2, random_state=42)
clf.fit(X_train_rand, y_train_rand)
leaky_preds = clf.predict_proba(X_test_rand)[:, 1]
leaky_p50 = precision_at_k(y_test_rand, leaky_preds)

# THE HONEST SPLIT (Grouped by Client)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))
clf.fit(X.iloc[train_idx], y.iloc[train_idx])
honest_preds = clf.predict_proba(X.iloc[test_idx])[:, 1]
honest_p50 = precision_at_k(y.iloc[test_idx], honest_preds)

print("=== SPLIT METHODOLOGY AUDIT ===")
print(f"Leaky Split (Random) Precision@50:  {leaky_p50:.3f}  <- Artificially inflated by memorizing client baselines.")
print(f"Honest Split (Grouped) Precision@50: {honest_p50:.3f}  <- The true, generalized performance.")

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*


**Audit Results: Clean.**


1. **Target Leakage:** The synthetic target `y` is computed dynamically. I verified that no columns containing `label`, `target`, or downstream performance flags exist within the `X` feature matrix.


2. **Client Identity Leakage:** The `synthetic_client_group` identifier was strictly passed to `GroupShuffleSplit` and excluded from model training to prevent the model from memorizing specific client conversion rates.


3. **Index Leakage:** Non-numeric identifiers (like `url` or `hash`) were programmatically dropped before feeding the matrix into the Random Forest classifier.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*


**Original (Dangerous) Claim:** "Our Random Forest model accurately predicts which pages are decaying and guarantees that refreshing them will recover lost search traffic."


**Audited (Safe) Claim:** "Our Random Forest model measures historical search telemetry to identify pages displaying directional indicators of content decay. It operates as a decision-support queue to help content teams prioritize editorial review, though causal traffic recovery relies on the quality of the subsequent human intervention."

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.